In [2]:
import os
import json
import pandas as pd

def is_valid_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            json.load(f)
        return True
    except json.JSONDecodeError as e:
        print(f"❌ JSON 文件损坏: {os.path.basename(path)} — {e}")
        return False

def safe_load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = f.read().strip()
        if raw.endswith(','):
            raw = raw[:-1]
        if not raw.endswith('}'):
            raw += '}'
        raw = raw.replace('\x00', '')
        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"❌ 修复失败: {path}, 错误: {e}")
            return None

def convert_labelme_to_coco(json_dir, output_json_name='instances.json'):
    # ==== 加载映射文件 ====
    brand_map_path = os.path.join(json_dir, 'image_brand_model_mapping.csv')
    label_map_path = os.path.join(json_dir, 'label_mapping.xlsx')

    if not os.path.exists(brand_map_path) or not os.path.exists(label_map_path):
        print(f"❌ 映射文件缺失: {brand_map_path} 或 {label_map_path}")
        return

    brand_df = pd.read_csv(brand_map_path)
    label_df = pd.read_excel(label_map_path)

    brand_model_to_id = dict(zip(brand_df['brand_model'], brand_df['brand_id']))
    label_to_id = dict(zip(label_df['label'], label_df['category_id']))

    images, annotations, categories = [], [], []
    image_id, annotation_id = 0, 0

    for label, cid in label_to_id.items():
        categories.append({'id': cid, 'name': label, 'supercategory': 'component'})

    for filename in os.listdir(json_dir):
        if not filename.endswith('.json') or filename.startswith('instances_'):
            continue
        if filename in ['brand_model_mapping.xlsx', 'label_mapping.xlsx']:
            continue

        json_path = os.path.join(json_dir, filename)
        if not is_valid_json(json_path):
            print(f"⚠️ 尝试自动修复 JSON：{filename}")

        data = safe_load_json(json_path)
        if not data:
            print(f"❌ 严重错误，跳过文件: {filename}")
            continue

        img_name = data.get('imagePath', filename.replace('.json', '.jpg'))
        height = data.get('imageHeight', 0)
        width = data.get('imageWidth', 0)
        brand_model = data.get('brand_model', None)

        if brand_model not in brand_model_to_id:
            print(f"⚠️ 跳过无 brand_model 映射的文件: {filename}")
            continue
        brand_id = brand_model_to_id[brand_model]

        # ✅ brand_id 写入 image，用于整图品牌分类
        images.append({
            'id': image_id,
            'file_name': img_name,
            'height': height,
            'width': width,
            'brand_id': brand_id
        })

        has_valid_annotation = False

        for shape in data.get('shapes', []):
            label = shape.get('label')
            if label not in label_to_id or 'points' not in shape:
                continue
            points = shape['points']
            try:
                x_coords = [p[0] for p in points]
                y_coords = [p[1] for p in points]
                x, y = min(x_coords), min(y_coords)
                w, h = max(x_coords) - x, max(y_coords) - y
            except Exception as e:
                print(f"⚠️ 跳过异常 shape: {e}")
                continue
            if w <= 0 or h <= 0:
                continue

            annotations.append({
                'id': annotation_id,
                'image_id': image_id,
                'category_id': label_to_id[label],
                'bbox': [x, y, w, h],
                'iscrowd': 0,
                'area': w * h,
                'brand_id': brand_id  # ✅ 可选写入 annotation，调试方便
            })
            annotation_id += 1
            has_valid_annotation = True

        if not has_valid_annotation:
            print(f"⚠️ 图片无有效标注（shapes），但仍保留图像项: {filename}")

        image_id += 1

    coco_output = {
        'images': images,
        'annotations': annotations,
        'categories': categories
    }

    output_path = os.path.join(json_dir, output_json_name)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, ensure_ascii=False, indent=2)

    print(f"✅ 写入完成: {output_path}")
    if is_valid_json(output_path):
        print("✅ 最终 COCO 文件结构验证通过")
    else:
        print("❌ 最终 COCO 文件结构仍然存在问题，请手动检查")

# === 主程序调用 ===
if __name__ == '__main__':
    train_dir = '/root/autodl-tmp/coco_dataset1/jsons/train'
    val_dir = '/root/autodl-tmp/coco_dataset1/jsons/val'

    convert_labelme_to_coco(train_dir, output_json_name='instances_train.json')
    convert_labelme_to_coco(val_dir, output_json_name='instances_val.json')


✅ 写入完成: /root/autodl-tmp/coco_dataset1/jsons/train/instances_train.json
✅ 最终 COCO 文件结构验证通过
✅ 写入完成: /root/autodl-tmp/coco_dataset1/jsons/val/instances_val.json
✅ 最终 COCO 文件结构验证通过


In [ ]:
pip install openpyxl